In [90]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)




data = pd.read_csv(r"C:\Users\samue\OneDrive\Desktop\ML Code\Proj1\aps+failure+at+scania+trucks\aps_failure_training_set.csv", skiprows=20)

In [96]:
#print(data.shape)
data = data.replace('na',np.nan) #Replace word 'na' with nan value'

#split data into data (x) and label (y)
x = data.drop(columns='class')
y = data['class']

#split data
x_train,x_val,y_train,y_val = train_test_split(x,y,test_size=0.2,random_state=5)#print(x.head(10))

#Impute x data & convert back to df
my_imputer = SimpleImputer(strategy='median',add_indicator=True)
imputed_x_train = pd.DataFrame(my_imputer.fit_transform(x_train))
imputed_x_val = pd.DataFrame(my_imputer.transform(x_val))

column_names = list(x_train.columns) #list of col names

for i in my_imputer.indicator_.features_: #each column name with atleast 1 nan value
    column_names.append(x_train.columns[i] + "_missing") #add that name + "_missing" to column name list


#Put col names back since imputer removed
imputed_x_train.columns = column_names
imputed_x_val.columns = column_names

#End of cell Results:
#X_train,x_val,y_train,y_valid ready to process


In [92]:
#create dummy model for baseline
print(y.value_counts())
dummy_outputs = y.value_counts().tolist()
dummy_neg = outputs[0]
dummy_y_pos = outputs[1]
dummy_y_pred = np.full(len(y_val), 'neg')

dummy_accuracy = accuracy_score(y_val, dummy_y_pred)
dummy_precision = precision_score(y_val, dummy_y_pred, pos_label='pos')
dummy_recall = recall_score(y_val, dummy_y_pred, pos_label='pos')
dummy_f1 = f1_score(y_val, dummy_y_pred, pos_label='pos')

print(f"Accuracy:   {dummy_accuracy:.4f}")
print(f"Precision:  {dummy_precision:.4f}")
print(f"Recall:     {dummy_recall:.4f}")
print(f"F1 Score:   {dummy_f1:.4f}")

dummy_stats = [dummy_accuracy,dummy_precision,dummy_recall,dummy_f1]


class
neg    59000
pos     1000
Name: count, dtype: int64
Accuracy:   0.9842
Precision:  0.0000
Recall:     0.0000
F1 Score:   0.0000


c:\Users\samue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
#create and train models

#Logisitc Regression
model_LogReg = LogisticRegression(class_weight={'neg': 1,'pos': 50},random_state=3)
model_LogReg.fit(imputed_x_train, y_train)

#Decision Tree
model_DTreeClass = DecisionTreeClassifier(class_weight={'neg': 1,'pos': 50},random_state=3)
model_DTreeClass.fit(imputed_x_train,y_train)

#Random Forest
#model_RndForest = RandomForestClassifier(class_weight={'neg': 1,'pos': 50},n_estimators=150, random_state=3,max_depth=20,min_samples_leaf=2)
model_RndForest = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight={'neg': 1, 'pos': 50},
    random_state=3,
    n_jobs=-1
)
model_RndForest.fit(imputed_x_train, y_train)

#XG Boost
model_XGBoost = XGBClassifier(
    scale_pos_weight = 50,
    n_estimators=150,
    random_state=3,
    max_depth=5
)
y_train_XG = y_train.map({'neg':0,'pos':1})
y_val_XG = y_val.map({'neg':0,'pos':1})
model_XGBoost.fit(imputed_x_train, y_train_XG)




c:\Users\samue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [129]:
#calculate metrics
def calculate_metrics(y_values,y_predictions,evaluation):
    accuracy = accuracy_score(y_values, y_predictions)
    precision = precision_score(y_values, y_predictions, pos_label=evaluation)
    recall = recall_score(y_values, y_predictions, pos_label=evaluation)
    f1 = f1_score(y_values, y_predictions, pos_label=evaluation)

    cm = confusion_matrix(y_values, y_predictions)
    tn, fp, fn, tp = cm.ravel()
    cost = 10*fp+500*fn

    vals = [
        ("cost", int(cost)),
        ("acc", round(float(accuracy), 2)),
        ("precision", round(float(precision), 2)),
        ("recall", round(float(recall), 2)),
        ("f1", round(float(f1), 2)),
        ("TP", int(tp)),
        ("FP", int(fp)),
        ("TN", int(tn)),
        ("FN", int(fn))
        ]
        
    print(vals)
    """
    print(f"COST: {cost}")
    print(f"Accuracy:   {accuracy:.4f}")
    print(f"Precision:  {precision:.4f}")
    print(f"Recall:     {recall:.4f}")
    print(f"F1 Score:   {f1:.4f}")
    print(f"TP: {tp}")
    print(f"FP: {fp}")
    print(f"TN: {tn}")
    print(f"FN: {fn}")
    """
    return cost

In [ ]:
#make predictions (thresh 50% default)
y_pred_LogReg = model_LogReg.predict(imputed_x_val)
y_pred_DecTree = model_DTreeClass.predict(imputed_x_val)
y_pred_RndForest = model_RndForest.predict(imputed_x_val)
y_pred_XGBoost = model_XGBoost.predict(imputed_x_val)



[0.10345298 0.13328353 0.11493893 ... 0.11346337 0.13340066 0.09811146]


In [ ]:
#Results for default threshold

models = [("model_LogReg",y_pred_LogReg,y_val,'pos'),("model_DTreeClass",y_pred_DecTree,y_val,'pos'),
("model_RndForest",y_pred_RndForest,y_val,'pos'),("model_XGBoost",y_pred_XGBoost,y_val_XG,1)]

for model in models:
    model_name,y_preds,y_vals,eval = model
    print(model_name+ ":")
    calculate_metrics(y_vals,y_preds,eval)

model_LogReg:
[('cost', 14880), ('acc', 0.96), ('precision', 0.28), ('recall', 0.89), ('f1', 0.42), ('TP', 169), ('FP', 438), ('TN', 11372), ('FN', 21)]
model_DTreeClass:
[('cost', 42610), ('acc', 0.99), ('precision', 0.63), ('recall', 0.56), ('f1', 0.59), ('TP', 106), ('FP', 61), ('TN', 11749), ('FN', 84)]
model_RndForest:
[('cost', 26360), ('acc', 0.99), ('precision', 0.79), ('recall', 0.73), ('f1', 0.76), ('TP', 138), ('FP', 36), ('TN', 11774), ('FN', 52)]
model_XGBoost:
[('cost', 17860), ('acc', 0.99), ('precision', 0.81), ('recall', 0.82), ('f1', 0.81), ('TP', 155), ('FP', 36), ('TN', 11774), ('FN', 35)]


In [150]:
#make probability predictions
pos_probs_LogReg = model_LogReg.predict_proba(imputed_x_val)[:,1]
pos_probs_DecTree = model_LogReg.predict_proba(imputed_x_val)[:,1]
pos_probs_RndForest = model_RndForest.predict_proba(imputed_x_val)[:,1]
pos_probs_XGBoost = model_XGBoost.predict_proba(imputed_x_val)[:,1]



In [155]:

threshold_options = np.linspace(0.001, 1, 1000)
scores = [[],[],[],[]]
for thresh in threshold_options:
    if thresh > 0.1:
        break
    
    print("\n\nThresh: " + str(thresh))
    y_predprob_LogReg = np.where(pos_probs_LogReg >= thresh, 'pos', 'neg')
    y_predprob_DecTree = np.where(pos_probs_DecTree >= thresh, 'pos', 'neg')
    y_predprob_RndForest = np.where(pos_probs_RndForest >= thresh, 'pos', 'neg')
    y_predprob_XGBoost = np.where(pos_probs_XGBoost >= thresh, 1, 0)

    print("Log Reg")
    cost1 = calculate_metrics(y_val,y_predprob_LogReg,'pos')
    print("Dec Tree")
    cost2 = calculate_metrics(y_val,y_predprob_DecTree,'pos')
    print("Rndm Forest")
    cost3 = calculate_metrics(y_val,y_predprob_RndForest,'pos')
    print("XG boost")
    cost4 = calculate_metrics(y_val_XG,y_predprob_XGBoost,1)

    scores[0].append(cost1)
    scores[1].append(cost2)
    scores[2].append(cost3)
    scores[3].append(cost4)

print("Lowest Cost for Log Reg: " + str(min(scores[0])) + ", Thresh = " + str(scores[0].index(min(scores[0]))*(0.001)+0.001))
print("Lowest Cost for Dec Tree: " + str(min(scores[1])) + ", Thresh = " + str(scores[1].index(min(scores[1]))*(0.001)+0.001))
print("Lowest Cost for Rnd Forest: " + str(min(scores[2])) + ", Thresh = " + str(scores[2].index(min(scores[2]))*(0.001)+0.001))
print("Lowest Cost for XGBoost: " + str(min(scores[3])) + ", Thresh = " + str(scores[3].index(min(scores[3]))*(0.001)+0.001))



Thresh: 0.001
Log Reg
[('cost', 118550), ('acc', 0.02), ('precision', 0.02), ('recall', 0.99), ('f1', 0.03), ('TP', 189), ('FP', 11805), ('TN', 5), ('FN', 1)]
Dec Tree
[('cost', 118550), ('acc', 0.02), ('precision', 0.02), ('recall', 0.99), ('f1', 0.03), ('TP', 189), ('FP', 11805), ('TN', 5), ('FN', 1)]
Rndm Forest
[('cost', 28060), ('acc', 0.78), ('precision', 0.07), ('recall', 0.98), ('f1', 0.12), ('TP', 187), ('FP', 2656), ('TN', 9154), ('FN', 3)]
XG boost
[('cost', 8090), ('acc', 0.98), ('precision', 0.41), ('recall', 0.94), ('f1', 0.57), ('TP', 179), ('FP', 259), ('TN', 11551), ('FN', 11)]


Thresh: 0.002
Log Reg
[('cost', 118550), ('acc', 0.02), ('precision', 0.02), ('recall', 0.99), ('f1', 0.03), ('TP', 189), ('FP', 11805), ('TN', 5), ('FN', 1)]
Dec Tree
[('cost', 118550), ('acc', 0.02), ('precision', 0.02), ('recall', 0.99), ('f1', 0.03), ('TP', 189), ('FP', 11805), ('TN', 5), ('FN', 1)]
Rndm Forest
[('cost', 25710), ('acc', 0.8), ('precision', 0.07), ('recall', 0.98), ('f1',

In [156]:
test_data = pd.read_csv(r"C:\Users\samue\OneDrive\Desktop\ML Code\Proj1\aps+failure+at+scania+trucks\aps_failure_test_set.csv",skiprows=20)
test_data = test_data.replace('na', np.nan)

x_test = test_data.drop(columns='class')
y_test = test_data['class']

imputed_x_test = pd.DataFrame(my_imputer.transform(x_test))
imputed_x_test.columns = column_names

y_test_XG = y_test.map({'neg':0,'pos':1})

my_thresh=0.006
pos_probs_XG_test = model_XGBoost.predict_proba(imputed_x_test)[:,1]
y_predprob_XG = np.where(pos_probs_XG_test >= my_thresh, 1, 0)

my_thresh1=0.089
pos_probs_EndForest_test = model_RndForest.predict_proba(imputed_x_test)[:,1]
y_predprob_RndFrst = np.where(pos_probs_EndForest_test >= my_thresh1, 'pos', 'neg')

cost_test = calculate_metrics(y_test_XG,y_predprob_XG,1)
print(cost_test)

cost_test = calculate_metrics(y_test,y_predprob_RndFrst,'pos')
print(cost_test)


[('cost', 11290), ('acc', 0.98), ('precision', 0.52), ('recall', 0.96), ('f1', 0.68), ('TP', 359), ('FP', 329), ('TN', 15296), ('FN', 16)]
11290
[('cost', 9680), ('acc', 0.96), ('precision', 0.37), ('recall', 0.98), ('f1', 0.54), ('TP', 368), ('FP', 618), ('TN', 15007), ('FN', 7)]
9680
